# Otimização de Infraestrutura da Envasadora Pneumática (Aqua Index): Árvores Geradoras Mínimas (MST)
## Dimensionamento Mínimo de Cabeamento SCADA e Malha Pneumática via Algoritmos de Kruskal e Prim

---

### 1. Contexto Industrial e Engenharia de Automação
Na montagem eletromecânica de uma **máquina pneumática de envasamento de água em copos** com mesa giratória indexada (**Aqua Index**), o roteamento de **eletrocalhas/chicotes de instrumentação SCADA** (sinais de sensores magnéticos, capacitivos, pressostato `PIT-301`, sensor de temperatura `TIT-401` e válvulas solenoides) e de **linhas pneumáticas** impacta diretamente o custo de capital (*CAPEX*), a imunidade a ruído eletromagnético e a facilidade de manutenção.

O objetivo da modelagem via **Matemática Discreta / Teoria dos Grafos e Árvores** é determinar a **Árvore Geradora Mínima (Minimum Spanning Tree - MST)**. A MST assegura conectividade total entre o painel central CLP/SCADA (`PLC-001`), a mesa indexadora (`MESA-001`), as cinco estações de processo (`E1` a `E5`) e a esteira de saída (`EST-501`), minimizando a metragem total de cabos e mangueiras sem criar laços redundantes.

#### Modelagem Matemática Formal
Dado um grafo não direcionado e ponderado $G = (V, E, w)$:
- $V$: Conjunto de vértices representando os **módulos e estações da envasadora** ($|V| = n$).
- $E$: Conjunto de arestas representando as **rotas físicas viáveis de eletrocalhas e tubulação pneumática** ($|E| = m$).
- $w: E \to \mathbb{R}^+$: Função de peso que atribui o comprimento físico em metros $w(e)$ a cada trecho $e \in E$.

Uma Árvore Geradora $T = (V, E_T)$ é um subgrafo conexo acíclico contendo todos os vértices de $V$, com $|E_T| = |V| - 1$. A MST busca minimizar a função objetivo:

$$W(T) = \sum_{e \in E_T} w(e)$$

---
### 2. Fundamentação dos Algoritmos

#### A. Algoritmo de Kruskal (Abordagem Centrada em Arestas)
- **Estratégia Gulosa (*Greedy*):** Ordena todas as arestas do grafo em ordem crescente de peso. Adiciona iterativamente o trecho mais curto à solução, desde que ele não forme ciclo com os trechos já selecionados.
- **Estrutura de Dados:** Utiliza **Conjuntos Disjuntos (*Disjoint Set Union / Union-Find*)** com otimizações de *Compressão de Caminho* e *União por Rank*.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |E|)$ ou $\mathcal{O}(|E| \log |V|)$.

#### B. Algoritmo de Prim (Abordagem Centrada em Vértices)
- **Estratégia Gulosa (*Greedy*):** Mantém uma única árvore conectada que cresce a partir de um vértice raiz $r \in V$ (no nosso projeto, o painel central `PLC-001`). A cada iteração, incorpora a aresta de menor peso que liga a árvore atual a um módulo ainda não conectado.
- **Estrutura de Dados:** Utiliza uma **Fila de Prioridades (*Min-Heap*)** para gerenciar a fronteira de corte.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |V|)$ com *Binary Heap* ou $\mathcal{O}(|E| + |V| \log |V|)$ com *Fibonacci Heap*.

In [ ]:
# Importação de Bibliotecas Standard e de Terceiros
from typing import List, Tuple, Dict, Set
import heapq
import math
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Bibliotecas carregadas com sucesso.")

---
### 3. Implementação Orientada a Objetos: Estruturas de Dados e Algoritmos

Abaixo definimos a classe `DisjointSet` (*Union-Find*) e as implementações puras dos algoritmos de **Kruskal** e **Prim** em Python 3 com anotações de tipo (*type hints*).

In [ ]:
class DisjointSet:
    """
    Estrutura de Dados Union-Find com otimizações de:
    - Compressão de Caminho (Path Compression)
    - União por Rank (Union by Rank)
    """
    def __init__(self, vertices: List[str]):
        self.parent: Dict[str, str] = {v: v for v in vertices}
        self.rank: Dict[str, int] = {v: 0 for v in vertices}

    def find(self, item: str) -> str:
        """Encontra o representante do conjunto com compressão de caminho."""
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, root1: str, root2: str) -> bool:
        """Une dois conjuntos. Retorna True se a união ocorreu, False se já pertenciam ao mesmo conjunto."""
        r1 = self.find(root1)
        r2 = self.find(root2)

        if r1 == r2:
            return False  # Formaria ciclo

        # União por Rank
        if self.rank[r1] < self.rank[r2]:
            self.parent[r1] = r2
        elif self.rank[r1] > self.rank[r2]:
            self.parent[r2] = r1
        else:
            self.parent[r2] = r1
            self.rank[r1] += 1
        return True


def kruskal_mst(vertices: List[str], edges: List[Tuple[str, str, float]]) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    """
    Executa o Algoritmo de Kruskal para obter a Árvore Geradora Mínima.
    Retorna as arestas escolhidas, o comprimento total e o histórico de passos.
    """
    # 1. Ordenação gulosa das arestas em ordem crescente de peso
    sorted_edges = sorted(edges, key=lambda x: x[2])
    dsu = DisjointSet(vertices)

    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []

    for u, v, w in sorted_edges:
        root_u = dsu.find(u)
        root_v = dsu.find(v)

        if root_u != root_v:
            dsu.union(u, v)
            mst_edges.append((u, v, w))
            total_weight += w
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Peso (m)": w,
                "Status": "INCLUÍDA",
                "Motivo": "Módulos em componentes separados"
            })
        else:
            execution_log.append({
                "Passo": len(execution_log) + 1,
                "Aresta": f"{u} - {v}",
                "Peso (m)": w,
                "Status": "REJEITADA",
                "Motivo": "Formaria ciclo (mesmo componente)"
            })

        if len(mst_edges) == len(vertices) - 1:
            # Parada antecipada: MST já possui |V| - 1 arestas
            break

    return mst_edges, round(total_weight, 2), execution_log


def prim_mst(vertices: List[str], edges: List[Tuple[str, str, float]], start_node: str) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    """
    Executa o Algoritmo de Prim utilizando Min-Heap (fila de prioridades).
    Retorna as arestas escolhidas, o comprimento total e o histórico de expansão.
    """
    # Construção da lista de adjacência
    adj: Dict[str, List[Tuple[float, str]]] = {v: [] for v in vertices}
    for u, v, w in edges:
        adj[u].append((w, v))
        adj[v].append((w, u))

    visited: Set[str] = {start_node}
    min_heap: List[Tuple[float, str, str]] = []  # (peso, origem, destino)

    # Adiciona arestas incidentes ao nó inicial
    for w, neighbor in adj[start_node]:
        heapq.heappush(min_heap, (w, start_node, neighbor))

    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []
    step = 1

    while min_heap and len(visited) < len(vertices):
        w, u, v = heapq.heappop(min_heap)

        if v in visited:
            execution_log.append({
                "Iteração": step,
                "Aresta Analisada": f"{u} - {v}",
                "Peso (m)": w,
                "Decisão": "IGNORADA",
                "Vértice Destino": v,
                "Motivo": f"{v} já está conectado à árvore"
            })
            step += 1
            continue

        visited.add(v)
        mst_edges.append((u, v, w))
        total_weight += w

        execution_log.append({
            "Iteração": step,
            "Aresta Analisada": f"{u} - {v}",
            "Peso (m)": w,
            "Decisão": "ADICIONADA",
            "Vértice Destino": v,
            "Motivo": f"Expande barramento para {v}"
        })
        step += 1

        # Adiciona novas arestas de corte a partir do módulo recém-conectado
        for next_w, neighbor in adj[v]:
            if neighbor not in visited:
                heapq.heappush(min_heap, (next_w, v, neighbor))

    return mst_edges, round(total_weight, 2), execution_log

---
### 4. Definição do Grafo da Máquina Envasadora de Água (Aqua Index)

Modelamos a célula automatizada da envasadora composta por **8 módulos operacionais** e **14 rotas físicas viáveis** de eletrocalhas/chicotes SCADA e alimentação pneumática (comprimentos em metros):

| Vértice | Módulo / Equipamento | Setor ISA-5.1 |
| :--- | :--- | :--- |
| `PLC-001` | Painel Central CLP/SCADA e Comandos Gerais | Setor 000 - Controle Geral |
| `MESA-001` | Servomotor e Encoder `SE-001` da Mesa Indexadora | Setor 000 - Indexação |
| `E1-DISP` | Estação 1: Dispensa de Copos (`XV-101` / `ZS-102`) | Setor 100 - Dispensa |
| `E2-ENV` | Estação 2: Envase e Dosagem 150 mL (`XV-202` / `XV-203`) | Setor 200 - Envase |
| `E3-TAMP` | Estação 3: Manipulador de Tampas (`XV-301` / `VAC-301` / `PIT-301`) | Setor 300 - Tampa |
| `E4-SEL` | Estação 4: Termosselagem (`XV-401` / `TIT-401`) | Setor 400 - Selagem |
| `E5-EJEC` | Estação 5: Ejeção Pneumática (`XV-501` / `XV-502`) | Setor 500 - Ejeção |
| `EST-501` | Esteira de Saída e Contagem de Produção | Setor 500 - Expedição |

In [ ]:
# Lista de Módulos da Envasadora (Vértices)
nodes = [
    "PLC-001", "MESA-001", "E1-DISP", "E2-ENV",
    "E3-TAMP", "E4-SEL", "E5-EJEC", "EST-501"
]

# Rotas Viáveis de Eletrocalhas/Cabos SCADA e Linhas Pneumáticas (Distâncias em Metros)
edges = [
    ("E5-EJEC", "EST-501", 1.1),
    ("E1-DISP", "E2-ENV", 1.2),
    ("E2-ENV", "E3-TAMP", 1.3),
    ("E3-TAMP", "E4-SEL", 1.4),
    ("E4-SEL", "E5-EJEC", 1.5),
    ("MESA-001", "E1-DISP", 1.6),
    ("MESA-001", "E2-ENV", 1.7),
    ("PLC-001", "E1-DISP", 1.8),
    ("MESA-001", "E3-TAMP", 1.9),
    ("PLC-001", "MESA-001", 2.0),
    ("MESA-001", "E4-SEL", 2.1),
    ("MESA-001", "E5-EJEC", 2.2),
    ("E5-EJEC", "E1-DISP", 2.6),
    ("PLC-001", "EST-501", 3.4)
]

total_network_length = round(sum(w for _, _, w in edges), 2)
print(f"Grafo da envasadora carregado: {len(nodes)} Módulos e {len(edges)} Rotas Viáveis ({total_network_length:.2f} m totais).")

---
### 5. Execução e Validação dos Algoritmos

In [ ]:
# Executando Kruskal
kruskal_edges, kruskal_weight, kruskal_log = kruskal_mst(nodes, edges)

# Executando Prim (partindo do Painel Central CLP/SCADA)
prim_edges, prim_weight, prim_log = prim_mst(nodes, edges, start_node="PLC-001")

economia_metros = round(total_network_length - kruskal_weight, 2)
economia_pct = (economia_metros / total_network_length) * 100

print("=" * 64)
print(" RESULTADOS DA OTIMIZAÇÃO TOPOLÓGICA — AQUA INDEX")
print("=" * 64)
print(f"Comprimento da Malha Completa:        {total_network_length:.2f} metros")
print(f"Metragem Mínima Otimizada (Kruskal):  {kruskal_weight:.2f} metros")
print(f"Metragem Mínima Otimizada (Prim):     {prim_weight:.2f} metros")
print(f"Equivalência Estrita de Custo:        {math.isclose(kruskal_weight, prim_weight)}")
print(f"Economia de Infraestrutura:           {economia_metros:.2f} m ({economia_pct:.1f}%)")
print("=" * 64)

#### Tabela 1: Histórico de Execução do Algoritmo de Kruskal

In [ ]:
df_kruskal = pd.DataFrame(kruskal_log)
display(df_kruskal)

#### Tabela 2: Histórico de Expansão do Algoritmo de Prim (Raiz: `PLC-001`)

In [ ]:
df_prim = pd.DataFrame(prim_log)
display(df_prim)

---
### 6. Visualização Gráfica da Envasadora e Topologia MST

O diagrama abaixo reproduz a geometria circular da mesa giratória indexada (`MESA-001`) cercada pelas cinco estações (`E1-DISP` a `E5-EJEC`), destacando em cinza tracejado as rotas redundantes descartadas e em verde contínuo o barramento ótimo da Árvore Geradora Mínima.

In [ ]:
# Criando o grafo NetworkX
G = nx.Graph()
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# Layout físico fiel à mesa indexadora da envasadora Aqua Index
pos = {
    "PLC-001": (0.0, 3.3),
    "MESA-001": (4.0, 2.5),
    "E1-DISP": (2.2, 3.9),
    "E2-ENV": (4.0, 4.9),
    "E3-TAMP": (5.8, 3.9),
    "E4-SEL": (5.8, 1.1),
    "E5-EJEC": (2.2, 1.1),
    "EST-501": (0.0, 0.9)
}

# Separação entre arestas da MST e rotas descartadas
mst_edge_pairs = set()
for u, v, w in kruskal_edges:
    mst_edge_pairs.add((u, v))
    mst_edge_pairs.add((v, u))

non_mst_edges = [e for e in G.edges() if (e[0], e[1]) not in mst_edge_pairs]
mst_edges_list = [e for e in G.edges() if (e[0], e[1]) in mst_edge_pairs]

fig, ax = plt.subplots(figsize=(14, 8))

# Cores distintas por função do nó
node_colors = [
    '#ff7f0e' if n == 'PLC-001' else '#2c3e50' if n == 'MESA-001' else '#1f77b4'
    for n in G.nodes()
]

# Desenhar vértices
nx.draw_networkx_nodes(G, pos, node_size=2800, node_color=node_colors, alpha=0.95, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', font_color='white', ax=ax)

# Desenhar rotas redundantes (descartadas)
nx.draw_networkx_edges(G, pos, edgelist=non_mst_edges, width=2, alpha=0.45, edge_color='gray', style='dashed', ax=ax)

# Desenhar rotas da MST (ativas)
nx.draw_networkx_edges(G, pos, edgelist=mst_edges_list, width=4.2, alpha=0.95, edge_color='#2ca02c', ax=ax)

# Rótulos de comprimentos das arestas
edge_labels = nx.get_edge_attributes(G, 'weight')
formatted_labels = {k: f"{v:.1f}m" for k, v in edge_labels.items()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=formatted_labels, font_size=9, font_color='#222222', ax=ax)

plt.title(
    f"Envasadora Pneumática Aqua Index — Topologia SCADA/Pneumática Otimizada via MST\n"
    f"Metragem Mínima Total: {kruskal_weight:.1f} m (Economia de {economia_pct:.1f}% vs. malha completa)",
    fontsize=13,
    fontweight='bold',
    pad=15
)
plt.axis('off')
plt.tight_layout()
plt.show()

---
### 7. Análise Técnica e Conclusões de Engenharia

1. **Redução de Custo de Capital (*CAPEX*) e Complexidade de Cabeamento:** A malha completa com todas as 14 rotas viáveis somava **25,8 metros**. A Árvore Geradora Mínima reduz o roteamento para apenas **9,9 metros** (economia de **61,6%** em eletrocalhas, cabos de sensores/solenoides e mangueiramento pneumático), mantendo 100% de conectividade entre todos os setores da máquina.
2. **Consistência Física com a Envasadora:** A solução ótima forma um barramento contínuo que sai do painel central (`PLC-001`), percorre sequencialmente o perímetro das estações (`E1-DISP` $\rightarrow$ `E2-ENV` $\rightarrow$ `E3-TAMP` $\rightarrow$ `E4-SEL` $\rightarrow$ `E5-EJEC` $\rightarrow$ `EST-501`) e faz uma única derivação radial curta até o centro da mesa (`MESA-001`), eliminando cruzamentos desnecessários sob a mesa giratória.
3. **Escolha do Algoritmo no Ciclo de Vida do Projeto:**
   - **Kruskal:** Recomendado na fase de projeto conceitual e lista de materiais (*BOM/FEED*), analisando todas as distâncias do *layout* CAD globalmente.
   - **Prim:** Recomendado durante a montagem física e o comissionamento, expandindo a infraestrutura continuamente a partir do Painel Central (`PLC-001`) estação por estação.